# K-Means Training: File-Based Approach (One CSV = One Movement)

This notebook implements **file-based K-means classification** where each CSV file represents **one complete movement action**.

## Data Structure Understanding:
- **Each CSV file** = **one complete movement** (takeoff, forward, landing, etc.)
- **All rows in a CSV file** = brain activity during that single movement action
- **Goal**: Classify which movement was being performed/thought about

## Why File-Based Approach:
- ✅ Matches motor imagery experimental design
- ✅ Prevents data leakage (related samples stay together)
- ✅ Each training sample = one complete mental action
- ✅ Standard practice in EEG motor imagery classification

> **Update the `data_root_processed` path below to point to your processed EEG data folder**

In [ ]:
# Imports and Setup
print("Starting K-Means File-Based Training...")

import sys
import os
import pathlib
import importlib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from scipy.signal import butter, sosfiltfilt, welch
from scipy.stats import skew, kurtosis

# Setup path to kmeans_model
_nb_dir = pathlib.Path.cwd()
_candidates = [
    _nb_dir,
    _nb_dir / "prediction_k_means" / "tensorflow",
    _nb_dir / "tensorflow",
]

_kmeans_dir = None
for cand in _candidates:
    if (cand / "kmeans_model.py").exists():
        _kmeans_dir = cand
        break

if _kmeans_dir and str(_kmeans_dir) not in sys.path:
    sys.path.insert(0, str(_kmeans_dir))

import kmeans_model
kmeans_model = importlib.reload(kmeans_model)

print("✓ Imports complete")

In [ ]:
# Configuration
print("Setting up configuration...")

# 🔧 UPDATE THIS PATH to your processed EEG data folder
data_root_processed = pathlib.Path("/home/elizabeth/Documents/Avatar-Elizabeth-Forked/Data_clean/Data_clean/processed")

# Expected movement folders
expected_labels = ["backward", "forward", "landing", "left", "right", "takeoff"]

# K-Means settings
n_clusters = 150
max_iter = 1000
tol = 1e-6
random_state = 42
use_pca = False
pca_components = 50

# Data processing settings
default_fs = 125.0  # Default sampling rate
ignore_files = {"remove-8channel-Report.csv"}

print(f"Data root: {data_root_processed}")
print(f"Expected movements: {expected_labels}")
print("✓ Configuration complete")

In [ ]:
# Load All CSV Files
print("Loading EEG data files...")

dfs = []
file_info = []

core_dir = pathlib.Path(data_root_processed)
if not core_dir.exists():
    raise FileNotFoundError(f"Data directory not found: {core_dir}")

# Load each movement folder
for label_dir in core_dir.iterdir():
    if not label_dir.is_dir() or label_dir.name not in expected_labels:
        continue

    print(f"Loading {label_dir.name} files...")
    for csv_file in label_dir.glob("*.csv"):
        if csv_file.name in ignore_files:
            continue

        try:
            df = pd.read_csv(csv_file, sep=",", on_bad_lines="skip")
            if df.empty or df.shape[1] < 2:
                continue

            df["src_filename"] = str(csv_file)
            df["movement_type"] = label_dir.name
            df["fs"] = default_fs

            dfs.append(df)
            file_info.append({
                "filename": csv_file.name,
                "movement": label_dir.name,
                "rows": len(df),
                "path": str(csv_file)
            })

        except Exception as e:
            print(f"Error loading {csv_file}: {e}")

# Combine all data
eeg_data = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
file_summary = pd.DataFrame(file_info)

print(f"✓ Loaded {len(dfs)} CSV files")
print(f"✓ Total rows: {len(eeg_data):,}")
print(f"✓ Combined shape: {eeg_data.shape}")

# Show file distribution
print("\nFile Distribution by Movement:")
movement_counts = file_summary.groupby("movement")["filename"].count()
for movement, count in movement_counts.items():
    print(f"  {movement:10s}: {count} files")

print(f"\nTotal files: {len(file_summary)}")

In [ ]:
# Data Quality Check
print("Performing data quality checks...")

# Check for missing values
missing_data = eeg_data.isnull().sum().sum()
print(f"Missing values: {missing_data}")

# Check sampling rates
fs_counts = eeg_data["fs"].value_counts()
print(f"Sampling rates: {dict(fs_counts)}")

# Check movement labels
movement_dist = eeg_data["movement_type"].value_counts()
print(f"Movement distribution (by rows):")
for movement, count in movement_dist.items():
    print(f"  {movement:10s}: {count:,} rows")

# Basic channel statistics
numeric_cols = eeg_data.select_dtypes(include=[np.number]).columns
eeg_cols = [c for c in numeric_cols if "EXG" in c]
print(f"EEG channels found: {len(eeg_cols)}")
print(f"Sample EEG channels: {eeg_cols[:5]}")

print("✓ Data quality checks complete")

In [ ]:
# Feature Engineering
print("Engineering features...")

# Select feature columns (exclude metadata)
drop_cols = ["src_filename", "movement_type", "fs"]
feature_cols = [c for c in eeg_data.columns if c not in drop_cols and c in numeric_cols]

# Clean features
features = eeg_data[feature_cols].copy()
features = features.replace([np.inf, -np.inf], np.nan)

# Handle missing values
if features.isnull().any().any():
    medians = features.median()
    features = features.fillna(medians)

# Remove zero-variance features
std_vals = features.std()
keep_cols = std_vals[std_vals > 0].index.tolist()
features = features[keep_cols]

print(f"✓ Feature matrix shape: {features.shape}")
print(f"✓ Features kept: {len(keep_cols)}")
print(f"✓ Features removed: {len(feature_cols) - len(keep_cols)}")

# Prepare labels
movement_labels = eeg_data["movement_type"]
label_encoder = {label: i for i, label in enumerate(sorted(movement_labels.unique()))}
numeric_labels = movement_labels.map(label_encoder)

# Add to dataframe for per-file processing
eeg_data["feature_cols"] = [keep_cols] * len(eeg_data)
eeg_data["numeric_label"] = numeric_labels

print(f"✓ Movement classes: {list(label_encoder.keys())}")
print(f"✓ Label encoding: {label_encoder}")

In [ ]:
# FILE-BASED FEATURE EXTRACTION
print("="*80)
print("FILE-BASED FEATURE EXTRACTION")
print("Each CSV file becomes ONE training sample")
print("="*80)

X_train_samples = []
y_train_samples = []
file_tracking = []

# Process each file individually
for file_path, file_data in eeg_data.groupby("src_filename"):
    movement_type = file_data["movement_type"].iloc[0]
    numeric_label = file_data["numeric_label"].iloc[0]
    sampling_rate = file_data["fs"].iloc[0]

    # Extract raw EEG and accelerometer data
    eeg_channels = [c for c in keep_cols if "EXG" in c]
    accel_channels = [c for c in keep_cols if "Accel" in c]

    eeg_data_raw = file_data[eeg_channels].values.astype(np.float32)
    accel_data_raw = file_data[accel_channels].values.astype(np.float32)

    # Apply bandpass filter to entire file
    if len(eeg_channels) > 0:
        eeg_filtered = kmeans_model.apply_bandpass_to_signal(eeg_data_raw, sampling_rate)
    else:
        eeg_filtered = np.array([]).reshape(len(file_data), 0)

    # Extract features from ENTIRE FILE
    file_features = kmeans_model.extract_window_features(eeg_filtered, accel_data_raw, sampling_rate)

    X_train_samples.append(file_features)
    y_train_samples.append(numeric_label)
    file_tracking.append({
        "file": file_path,
        "movement": movement_type,
        "samples": len(file_data),
        "duration_sec": len(file_data) / sampling_rate
    })

# Convert to numpy arrays
X = np.array(X_train_samples, dtype=np.float32)
y = np.array(y_train_samples, dtype=np.int32)

print(f"✓ Created {X.shape[0]} training samples (one per CSV file)")
print(f"✓ Each sample has {X.shape[1]} features")
print(f"✓ No data leakage - files stay intact")

# Show sample distribution
tracking_df = pd.DataFrame(file_tracking)
print(f"\nSample Distribution:")
for movement in sorted(tracking_df["movement"].unique()):
    count = (tracking_df["movement"] == movement).sum()
    avg_duration = tracking_df[tracking_df["movement"] == movement]["duration_sec"].mean()
    print(f"  {movement:10s}: {count:3d} files, avg {avg_duration:.1f}s each")

print(f"\nTotal training samples: {len(X)}")
print("="*80)

In [ ]:
# Train/Test Split
print("Splitting data into train/test sets...")

# Stratified split to maintain movement balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=random_state,
    stratify=y
)

print(f"✓ Train set: {X_train.shape[0]} samples")
print(f"✓ Test set:  {X_test.shape[0]} samples")
print(f"✓ Features:  {X_train.shape[1]}")

# Show class distribution
train_labels, train_counts = np.unique(y_train, return_counts=True)
test_labels, test_counts = np.unique(y_test, return_counts=True)

movement_names = list(label_encoder.keys())
print(f"\nTrain set distribution:")
for label, count in zip(train_labels, train_counts):
    movement = movement_names[label]
    print(f"  {movement:10s}: {count} samples")

print(f"\nTest set distribution:")
for label, count in zip(test_labels, test_counts):
    movement = movement_names[label]
    print(f"  {movement:10s}: {count} samples")

In [ ]:
# Feature Scaling
print("Scaling features...")

# Calculate scaling parameters from training data only
train_mean = np.nanmean(X_train, axis=0)
train_std = np.nanstd(X_train, axis=0)

# Avoid division by zero
train_std = np.where(train_std == 0, 1.0, train_std)
train_mean = np.where(np.isnan(train_mean), 0.0, train_mean)

# Apply scaling
X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

# Handle any remaining NaN or inf values
X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

print(f"✓ Training data scaled: {X_train_scaled.shape}")
print(f"✓ Test data scaled: {X_test_scaled.shape}")
print(f"✓ Scaling parameters computed from training set only")

In [ ]:
# Optional PCA
print("Applying PCA (if enabled)...")

if use_pca:
    pca = PCA(n_components=pca_components, random_state=random_state)
    X_train_final = pca.fit_transform(X_train_scaled)
    X_test_final = pca.transform(X_test_scaled)

    explained_var = np.sum(pca.explained_variance_ratio_) * 100
    print(f"✓ PCA applied: {X_train_final.shape[1]} components")
    print(f"✓ Variance retained: {explained_var:.1f}%")
else:
    X_train_final = X_train_scaled
    X_test_final = X_test_scaled
    pca = None
    print("✓ PCA disabled - using scaled features directly")

print(f"✓ Final training shape: {X_train_final.shape}")
print(f"✓ Final test shape: {X_test_final.shape}")

In [ ]:
# Train K-Means Model
print("="*60)
print("TRAINING K-MEANS MODEL")
print("="*60)

model = kmeans_model.KMeansTF(
    n_clusters=n_clusters,
    max_iter=max_iter,
    tol=tol,
    random_state=random_state
)

print(f"Starting training on {X_train_final.shape[0]} samples with {X_train_final.shape[1]} features...")
print(f"Model settings: {n_clusters} clusters, max_iter={max_iter}, tol={tol}")

model.fit(X_train_final, y_train)

print(f"✓ Training complete!")
print(f"✓ Final inertia: {model.inertia_:.2f}")
print(f"✓ Converged in {model.n_iter_} iterations")

# Show cluster-to-label mapping
print(f"\nCluster-to-Movement Mapping:")
movement_names = list(label_encoder.keys())
for cluster_id, label_id in sorted(model.cluster_to_label_map_.items()):
    movement = movement_names[label_id]
    print(f"  Cluster {cluster_id:3d} → {movement}")

In [ ]:
# Evaluate Model
print("="*60)
print("MODEL EVALUATION")
print("="*60)

# Make predictions
y_pred = model.predict(X_test_final)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"✓ Overall Accuracy: {accuracy:.1%}")

# Confusion matrix
print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print("Predicted →")
print("Actual ↓")
for i, actual_movement in enumerate(movement_names):
    row = "  ".join("2d" if j == i else "2d" for j in range(len(movement_names)))
    print(f"{actual_movement:10s}: {row}")

# Classification report
print(f"\nDetailed Classification Report:")
report = classification_report(y_test, y_pred, target_names=movement_names, zero_division=0)
print(report)

In [ ]:
# Confidence Analysis
print("="*60)
print("CONFIDENCE ANALYSIS")
print("="*60)

# Get prediction probabilities
y_pred_proba = model.predict_proba(X_test_final)

print("Movement Confidence Analysis:")
print("-" * 40)

for i, movement in enumerate(movement_names):
    # Confidence in predicting this movement
    movement_confidence = y_pred_proba[:, i]
    avg_conf = movement_confidence.mean()
    max_conf = movement_confidence.max()
    min_conf = movement_confidence.min()

    print(f"{movement:10s}:")
    print(f"  Average: {avg_conf:.2%}")
    print(f"  Maximum: {max_conf:.2%}")
    print(f"  Minimum: {min_conf:.2%}")
    print()

# Overall confidence distribution
all_confidences = y_pred_proba.max(axis=1)  # Max confidence for each prediction
print(f"Overall Prediction Confidence:")
print(f"  Average: {all_confidences.mean():.2%}")
print(f"  90th percentile: {np.percentile(all_confidences, 90):.2%}")
print(f"  50th percentile: {np.percentile(all_confidences, 50):.2%}")
print(f"  10th percentile: {np.percentile(all_confidences, 10):.2%}")

# Show most/least confident predictions
most_confident_idx = np.argmax(all_confidences)
least_confident_idx = np.argmin(all_confidences)

print(f"\nMost confident prediction:")
print(f"  True: {movement_names[y_test[most_confident_idx]]}")
print(f"  Predicted: {movement_names[y_pred[most_confident_idx]]}")
print(f"  Confidence: {all_confidences[most_confident_idx]:.2%}")

print(f"\nLeast confident prediction:")
print(f"  True: {movement_names[y_test[least_confident_idx]]}")
print(f"  Predicted: {movement_names[y_pred[least_confident_idx]]}")
print(f"  Confidence: {all_confidences[least_confident_idx]:.2%}")


In [ ]:
# Save Model
print("="*60)
print("SAVING MODEL")
print("="*60)

# Prepare metadata
metadata = {
    "model_type": "file_based_kmeans",
    "n_clusters": n_clusters,
    "feature_columns": keep_cols,
    "movement_classes": movement_names,
    "label_encoder": label_encoder,
    "scaling_mean": train_mean.tolist(),
    "scaling_std": train_std.tolist(),
    "pca": pca,
    "use_pca": use_pca,
    "training_samples": len(X_train),
    "test_samples": len(X_test),
    "accuracy": float(accuracy),
    "data_root": str(data_root_processed),
    "created_at": pd.Timestamp.now().isoformat()
}

# Save model
model_filename = "kmeans_file_based_model.pth"
kmeans_model.save_model(model_filename, model, metadata)

print(f"✓ Model saved as: {model_filename}")
print(f"✓ Metadata included: {list(metadata.keys())}")
print(f"✓ Ready for inference on new movement data")

# Show how to load for inference
print(f"\nTo load this model for prediction:")
print(f"model, meta = kmeans_model.load_model('{model_filename}')")
print(f"prediction = model.predict(new_features)")

# Summary

## What This Notebook Does:
- **File-Based Classification**: Each CSV file = one complete movement action
- **Motor Imagery Focus**: Classifies brain activity during movement thoughts
- **Data Integrity**: No data leakage between train/test sets

## Key Results:
- **Training Samples**: {X_train.shape[0]} (one per CSV file)
- **Test Samples**: {X_test.shape[0]}
- **Accuracy**: {accuracy:.1%}
- **Features**: {X_train.shape[1]} per sample

## Movement Classes:
{movement_names}

## Next Steps:
1. **Tune Parameters**: Adjust `n_clusters`, try PCA, modify features
2. **Cross-Validation**: Use k-fold CV for more robust evaluation
3. **Feature Engineering**: Experiment with different frequency bands or temporal features
4. **Real-time Deployment**: Use saved model for live movement classification

## File Structure:
- **Input**: CSV files in movement folders (takeoff/, forward/, etc.)
- **Output**: Trained K-means model + preprocessing metadata
- **Each CSV**: One complete movement recording